In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from pyproj import Transformer
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
import lightgbm as lgb
from lightgbm import LGBMRegressor

In [2]:
%store -r cn0dbhz_merge_df
%store -r satcount_merge_df
%store -r elevationdeg_merge_df
%store -r acc_merge_df
%store -r gyro_merge_df
# %store -r energy_merge_df

%store -r satcountdelta_merge_df
# %store -r energydelta_merge_df
%store -r cn0dbhzdelta_merge_df
%store -r cn0dbhzmean_merge_df
# %store -r energystd_merge_df
# %store -r clarity_merge_df

In [3]:
merge_df = cn0dbhz_merge_df.merge(satcount_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'SatCount']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(elevationdeg_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'SvElevationDegrees']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(acc_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'AccMag']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(gyro_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'GyroMag']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
# merge_df = merge_df.merge(energy_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'TotalMotionEnergy']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')

merge_df = merge_df.merge(satcountdelta_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'SatCountDelta']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(cn0dbhzdelta_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'Cn0DbHzDelta']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
# merge_df = merge_df.merge(energydelta_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'EnergyDelta']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
merge_df = merge_df.merge(cn0dbhzmean_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'Cn0DbHzMean']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
# merge_df = merge_df.merge(energystd_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'EnergyStd']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')
# merge_df = merge_df.merge(clarity_merge_df[['UnixTimeMillis', 'drive_id', 'device', 'SignalClarity']], on = ['UnixTimeMillis', 'device', 'drive_id'], how = 'left')

In [4]:
print(merge_df.isna().sum())

drive_id              0
device                0
UnixTimeMillis        0
Cn0DbHz               0
ErrorXEcefMeters      0
ErrorYEcefMeters      0
ErrorZEcefMeters      0
SatCount              0
SvElevationDegrees    1
AccMag                0
GyroMag               0
SatCountDelta         0
Cn0DbHzDelta          0
Cn0DbHzMean           0
dtype: int64


In [5]:
# drop null values

# merge_df = merge_df.dropna(subset=['SvElevationDegrees', 'TotalMotionEnergy'])
merge_df = merge_df.dropna(subset=['SvElevationDegrees'])

print(merge_df.isna().sum())
print()
print('Merge Shape: ')
print(merge_df.shape)
print()

drive_id              0
device                0
UnixTimeMillis        0
Cn0DbHz               0
ErrorXEcefMeters      0
ErrorYEcefMeters      0
ErrorZEcefMeters      0
SatCount              0
SvElevationDegrees    0
AccMag                0
GyroMag               0
SatCountDelta         0
Cn0DbHzDelta          0
Cn0DbHzMean           0
dtype: int64

Merge Shape: 
(153630, 14)



In [56]:
X = merge_df[['Cn0DbHz', 'SatCount', 'SvElevationDegrees', 'AccMag', 'GyroMag', 'SatCountDelta', 'Cn0DbHzDelta', 'Cn0DbHzMean']]
y = pd.Series(merge_df['ErrorXEcefMeters'])

In [57]:
# split train and test data with 80% and 20% respectively

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [58]:
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [59]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'linear_tree': True,      # <-- THIS IS THE EXTRAPOLATION FIX
    'learning_rate': 0.05,
    'num_leaves': 15,
    'verbosity': -1
}

num_round = 2000 
callbacks = [
    lgb.early_stopping(stopping_rounds=20), # Increased to 20 to give it more "patience"
    lgb.log_evaluation(period=50)           # Prints progress every 50 rounds
]

modelx = lgb.train(
    params, 
    train_data, 
    num_boost_round=num_round, 
    valid_sets=[test_data],
    callbacks=callbacks
)

Training until validation scores don't improve for 20 rounds
[50]	valid_0's rmse: 2.42155
[100]	valid_0's rmse: 2.39438
[150]	valid_0's rmse: 2.37995
[200]	valid_0's rmse: 2.37021
[250]	valid_0's rmse: 2.35877
[300]	valid_0's rmse: 2.35156
[350]	valid_0's rmse: 2.34287
[400]	valid_0's rmse: 2.3382
[450]	valid_0's rmse: 2.3328
[500]	valid_0's rmse: 2.32734
[550]	valid_0's rmse: 2.32135
[600]	valid_0's rmse: 2.31959
[650]	valid_0's rmse: 2.31661
[700]	valid_0's rmse: 2.31501
[750]	valid_0's rmse: 2.31283
[800]	valid_0's rmse: 2.31066
[850]	valid_0's rmse: 2.30915
[900]	valid_0's rmse: 2.30778
[950]	valid_0's rmse: 2.30626
[1000]	valid_0's rmse: 2.30537
[1050]	valid_0's rmse: 2.30395
[1100]	valid_0's rmse: 2.30237
[1150]	valid_0's rmse: 2.30143
[1200]	valid_0's rmse: 2.30085
Early stopping, best iteration is:
[1213]	valid_0's rmse: 2.30056


In [60]:
# modelx = LGBMRegressor(metric='rmse')
# modelx.fit(X_train, y_train)

# Y_train = modelx.predict(X_train)
# Y_test = modelx.predict(X_test)

# print("Training RMSE: ", np.sqrt(mean_squared_error(y_train, Y_train)))
# print("Validation RMSE: ", np.sqrt(mean_squared_error(y_test, Y_test)))

Y_train = modelx.predict(X_train)
Y_test = modelx.predict(X_test)

print("LightGBM Training RMSE: ", np.sqrt(mean_squared_error(y_train, Y_train)))
print("LightGBM Validation RMSE: ", np.sqrt(mean_squared_error(y_test, Y_test)))

LightGBM Training RMSE:  2.167330482697382
LightGBM Validation RMSE:  2.3005581848201997


In [61]:
X = merge_df[['Cn0DbHz', 'SatCount', 'SvElevationDegrees', 'AccMag', 'GyroMag', 'SatCountDelta', 'Cn0DbHzDelta', 'Cn0DbHzMean']]
y = pd.Series(merge_df['ErrorYEcefMeters'])

In [62]:
# split train and test data with 80% and 20% respectively

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [63]:
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [64]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'linear_tree': True,      # <-- THIS IS THE EXTRAPOLATION FIX
    'learning_rate': 0.05,
    'num_leaves': 15,
    'verbosity': -1
}

num_round = 2000 
callbacks = [
    lgb.early_stopping(stopping_rounds=20), # Increased to 20 to give it more "patience"
    lgb.log_evaluation(period=50)           # Prints progress every 50 rounds
]

modely = lgb.train(
    params, 
    train_data, 
    num_boost_round=num_round, 
    valid_sets=[test_data],
    callbacks=callbacks
)

Training until validation scores don't improve for 20 rounds
[50]	valid_0's rmse: 3.07851
[100]	valid_0's rmse: 3.04029
[150]	valid_0's rmse: 3.0178
[200]	valid_0's rmse: 2.99999
[250]	valid_0's rmse: 2.98324
[300]	valid_0's rmse: 2.97036
[350]	valid_0's rmse: 2.96051
[400]	valid_0's rmse: 2.94987
[450]	valid_0's rmse: 2.94495
[500]	valid_0's rmse: 2.93928
[550]	valid_0's rmse: 2.93538
[600]	valid_0's rmse: 2.93036
[650]	valid_0's rmse: 2.92719
[700]	valid_0's rmse: 2.92396
[750]	valid_0's rmse: 2.92065
[800]	valid_0's rmse: 2.91832
[850]	valid_0's rmse: 2.91566
[900]	valid_0's rmse: 2.91304
[950]	valid_0's rmse: 2.91062
[1000]	valid_0's rmse: 2.90781
[1050]	valid_0's rmse: 2.90512
[1100]	valid_0's rmse: 2.90391
[1150]	valid_0's rmse: 2.90291
Early stopping, best iteration is:
[1162]	valid_0's rmse: 2.90251


In [65]:
# modely = LGBMRegressor(metric='rmse')
# modely.fit(X_train, y_train)

# Y_train = modely.predict(X_train)
# Y_test = modely.predict(X_test)

# print("Training RMSE: ", np.sqrt(mean_squared_error(y_train, Y_train)))
# print("Validation RMSE: ", np.sqrt(mean_squared_error(y_test, Y_test)))

Y_train = modely.predict(X_train)
Y_test = modely.predict(X_test)

print("LightGBM Training RMSE: ", np.sqrt(mean_squared_error(y_train, Y_train)))
print("LightGBM Validation RMSE: ", np.sqrt(mean_squared_error(y_test, Y_test)))

LightGBM Training RMSE:  2.758284042229605
LightGBM Validation RMSE:  2.9025082321926727


In [66]:
X = merge_df[['Cn0DbHz', 'SatCount', 'SvElevationDegrees', 'AccMag', 'GyroMag', 'SatCountDelta', 'Cn0DbHzDelta', 'Cn0DbHzMean']]
y = pd.Series(merge_df['ErrorZEcefMeters'])

In [67]:
# split train and test data with 80% and 20% respectively

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [68]:
train_data = lgb.Dataset(X_train, label=y_train)
test_data = lgb.Dataset(X_test, label=y_test, reference=train_data)

In [69]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'linear_tree': True,      # <-- THIS IS THE EXTRAPOLATION FIX
    'learning_rate': 0.05,
    'num_leaves': 15,
    'verbosity': -1
}

num_round = 2000 
callbacks = [
    lgb.early_stopping(stopping_rounds=20), # Increased to 20 to give it more "patience"
    lgb.log_evaluation(period=50)           # Prints progress every 50 rounds
]

modelz = lgb.train(
    params, 
    train_data, 
    num_boost_round=num_round, 
    valid_sets=[test_data],
    callbacks=callbacks
)

Training until validation scores don't improve for 20 rounds
[50]	valid_0's rmse: 2.97145
[100]	valid_0's rmse: 2.92959
[150]	valid_0's rmse: 2.91234
[200]	valid_0's rmse: 2.89814
[250]	valid_0's rmse: 2.88741
[300]	valid_0's rmse: 2.87711
[350]	valid_0's rmse: 2.86555
[400]	valid_0's rmse: 2.85822
[450]	valid_0's rmse: 2.8473
[500]	valid_0's rmse: 2.84341
[550]	valid_0's rmse: 2.83791
[600]	valid_0's rmse: 2.83381
[650]	valid_0's rmse: 2.82766
[700]	valid_0's rmse: 2.82297
[750]	valid_0's rmse: 2.81926
Early stopping, best iteration is:
[771]	valid_0's rmse: 2.81761


In [70]:
# modelz = LGBMRegressor(metric='rmse')
# modelz.fit(X_train, y_train)

# Y_train = modelz.predict(X_train)
# Y_test = modelz.predict(X_test)

# print("Training RMSE: ", np.sqrt(mean_squared_error(y_train, Y_train)))
# print("Validation RMSE: ", np.sqrt(mean_squared_error(y_test, Y_test)))

Y_train = modelz.predict(X_train)
Y_test = modelz.predict(X_test)

print("LightGBM Training RMSE: ", np.sqrt(mean_squared_error(y_train, Y_train)))
print("LightGBM Validation RMSE: ", np.sqrt(mean_squared_error(y_test, Y_test)))

LightGBM Training RMSE:  2.699000315480384
LightGBM Validation RMSE:  2.817614672048332


In [46]:
%store -r merge_test_df

In [47]:
print(merge_test_df.isna().sum())

UnixTimeMillis            0
Cn0DbHz                   0
SatCount                  0
SvElevationDegrees        0
tripId                    0
AccMag                    0
GyroMag                   0
SatCountDelta             0
Cn0DbHzDelta              0
Cn0DbHzMean               0
WlsPositionXEcefMeters    0
WlsPositionYEcefMeters    0
WlsPositionZEcefMeters    0
dtype: int64


In [71]:
features_test_df = merge_test_df.drop(columns=['UnixTimeMillis', 'tripId', 'WlsPositionXEcefMeters','WlsPositionYEcefMeters', 'WlsPositionZEcefMeters'])

predicted_errorx = modelx.predict(features_test_df)
predicted_errory = modely.predict(features_test_df)
predicted_errorz = modelz.predict(features_test_df)

print('predicted_errorx: ')
print(predicted_errorx)
print()
print('predicted_errory: ')
print(predicted_errory)
print()
print('predicted_errorz: ')
print(predicted_errorz)
print()

predicted_errorx: 
[-0.84233551 -0.3468186  -0.44458431 ... -0.94219384 -0.51921225
 -1.3465743 ]

predicted_errory: 
[-0.39759754 -0.88651644 -0.93926286 ... -1.27627541 -1.95709396
 -2.60683752]

predicted_errorz: 
[ 0.33093824  0.23121237  0.23725723 ... -0.25861054  0.52386879
  0.09845511]



In [72]:
submission_ecef = pd.DataFrame()
submission_ecef['MeasurementX_Corr'] = merge_test_df['WlsPositionXEcefMeters'] + predicted_errorx
submission_ecef['MeasurementY_Corr'] = merge_test_df['WlsPositionYEcefMeters'] + predicted_errory
submission_ecef['MeasurementZ_Corr'] = merge_test_df['WlsPositionZEcefMeters'] + predicted_errorz

In [73]:
transformer = Transformer.from_crs("EPSG:4978", "EPSG:4326", always_xy=True)

lon, lat, alt = transformer.transform(
    submission_ecef['MeasurementX_Corr'].values, 
    submission_ecef['MeasurementY_Corr'].values,
    submission_ecef['MeasurementZ_Corr'].values,
)

In [74]:
submission = pd.DataFrame()
submission['tripId'] = merge_test_df['tripId']
submission['UnixTimeMillis'] = merge_test_df['UnixTimeMillis']
submission['LatitudeDegrees'] = lat
submission['LongitudeDegrees'] = lon

print(submission)
print()

                                          tripId  UnixTimeMillis  \
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650832999   
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650833999   
2      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650834999   
3      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650835999   
4      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650836999   
...                                          ...             ...   
66092           2022-04-25-US-OAK-2/GooglePixel4   1650927742650   
66093           2022-04-25-US-OAK-2/GooglePixel4   1650927743642   
66094           2022-04-25-US-OAK-2/GooglePixel4   1650927744651   
66095           2022-04-25-US-OAK-2/GooglePixel4   1650927745640   
66096           2022-04-25-US-OAK-2/GooglePixel4   1650927746632   

       LatitudeDegrees  LongitudeDegrees  
0            37.395826       -122.102973  
1            37.395847       -122.102984  
2            37.395831       -122.102999  
3          

In [75]:
def remove_spikes(df):
    dist_lat = df.groupby('tripId')['LatitudeDegrees'].diff().abs()
    dist_lng = df.groupby('tripId')['LongitudeDegrees'].diff().abs()
    
    mask = (dist_lat > 0.0005) | (dist_lng > 0.0005)
    
    df.loc[mask, ['LatitudeDegrees', 'LongitudeDegrees']] = np.nan
    
    return df.groupby('tripId').apply(lambda group: group.interpolate())

In [76]:
def apply_savgol_filter(df, window=11, poly=3):
    df = df.reset_index(drop=True)
    df_out = df.copy()
    
    for trip_id, trip_df in df_out.groupby('tripId'):
        
        if len(trip_df) > window:
            df_out.loc[trip_df.index, 'LatitudeDegrees'] = savgol_filter(trip_df['LatitudeDegrees'], window, poly)
            df_out.loc[trip_df.index, 'LongitudeDegrees'] = savgol_filter(trip_df['LongitudeDegrees'], window, poly)

    return df_out

In [77]:
submission = remove_spikes(submission)
submission = apply_savgol_filter(submission)

/var/folders/w8/d9x92py92j792gw_1kyfc8sh0000gn/T/ipykernel_19305/2132394206.py:9: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  return df.groupby('tripId').apply(lambda group: group.interpolate())
/var/folders/w8/d9x92py92j792gw_1kyfc8sh0000gn/T/ipykernel_19305/2132394206.py:9: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  return df.groupby('tripId').apply(lambda group: group.interpolate())
/var/folders/w8/d9x92py92j792gw_1kyfc8sh0000gn/T/ipykernel_19305/2132394206.py:9: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  return df.groupby('tripId').apply(lambda group: group.interpolate())
/var/folders/w8/d9x92py92j792gw

In [78]:
print(submission)
print()
submission.to_csv('./SUBMISSIONS/LIGHT_GBM/submission.csv', index=False)

                                          tripId  UnixTimeMillis  \
0      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650832999   
1      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650833999   
2      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650834999   
3      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650835999   
4      2021-04-28-US-MTV-2/SamsungGalaxyS20Ultra   1619650836999   
...                                          ...             ...   
66092           2022-04-25-US-OAK-2/GooglePixel4   1650927742650   
66093           2022-04-25-US-OAK-2/GooglePixel4   1650927743642   
66094           2022-04-25-US-OAK-2/GooglePixel4   1650927744651   
66095           2022-04-25-US-OAK-2/GooglePixel4   1650927745640   
66096           2022-04-25-US-OAK-2/GooglePixel4   1650927746632   

       LatitudeDegrees  LongitudeDegrees  
0            37.395826       -122.102974  
1            37.395834       -122.102979  
2            37.395845       -122.102983  
3          